In [ ]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/expenses.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["day_name"] = df["date"].dt.day_name()

df.groupby("date")["amount"].sum()

In [ ]:
df["amount"].sum()

In [ ]:
df.groupby("category")["amount"].sum().sort_values(ascending=False)

In [ ]:
df.groupby("category")["amount"].sum().plot(kind="bar")
plt.show()

In [ ]:
df.groupby("day_name")["amount"].sum()

In [ ]:
df.groupby("month")["amount"].sum()

In [ ]:
df['amount'].mean()

In [ ]:
mean = df["amount"].mean()
std = df["amount"].std()

threshold = mean + 2 * std

df[df["amount"] > threshold]

In [ ]:
df['date'] = pd.to_datetime(df['date'])

# Feature engineering
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek

daily_df = df.groupby('date')['amount'].sum().reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['day'] = daily_df['date'].dt.day
daily_df['month'] = daily_df['date'].dt.month
daily_df['day_of_week'] = daily_df['date'].dt.dayofweek

X = daily_df[['day', 'month', 'day_of_week']]
y = daily_df['amount']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

In [ ]:
daily_df['is_weekend'] = daily_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
daily_df['rolling_avg'] = daily_df['amount'].rolling(window=2).mean()
daily_df = daily_df.dropna()

X = daily_df[['day', 'month', 'day_of_week', 'is_weekend', 'rolling_avg']]
y = daily_df['amount']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

In [ ]:
from sklearn.ensemble import IsolationForest

df = pd.read_csv('../data/expenses.csv')

X = df[['amount']]

model = IsolationForest(contamination=0.03)
model.fit(X)

df['anomaly'] = model.predict(X)
df[df['anomaly'] == -1]

In [ ]:
df[df['anomaly'] == -1].groupby('category')['amount'].count()

In [ ]:
def generate_insights(df):
    print("📊 ===== EXPENSE INSIGHTS =====\n")
    
    # Top category
    top_category = df.groupby('category')['amount'].sum().idxmax()
    print(f"🔹 Top Category: {top_category}")
    
    # Average spend
    avg_spend = df['amount'].mean()
    print(f"🔹 Average Spend: ₹{avg_spend:.2f}")
    
    # Total spend
    total_spend = df['amount'].sum()
    print(f"🔹 Total Spend: ₹{total_spend}")
    
    # Anomalies
    anomalies = df[df['anomaly'] == -1]
    print(f"\n🚨 Anomalies Detected: {len(anomalies)}")
    
    print(anomalies[['date', 'amount', 'category']])
    
    
generate_insights(df)

In [ ]:
from sklearn.ensemble import IsolationForest

df = pd.read_csv('../data/expenses.csv')

df_food = df[df['category'] == 'Food']

X = df_food[['amount']]

model = IsolationForest(contamination=0.01)
model.fit(X)

df_food['anomaly'] = model.predict(X)
df_food[df_food['anomaly'] == -1]

In [ ]:
df.sort_values(by='amount', ascending=False).head(3)

In [ ]:
import pandas as pd

df = pd.read_csv('../data/expenses.csv')
df.head()

In [ ]:
# Convert date
df['date'] = pd.to_datetime(df['date'])

# Extract features
df['day_of_week'] = df['date'].dt.dayofweek

# Weekend flag
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

df.head()

In [ ]:
df_encoded = pd.get_dummies(df, columns=['category'])
df_encoded.head()

In [ ]:
X = df_encoded.drop(['date', 'note'], axis=1)
X.head()

In [ ]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(contamination=0.03, random_state=42)
model.fit(X)

In [ ]:
df['anomaly'] = model.predict(X)

df['anomaly'].value_counts()

In [ ]:
def generate_insights(df):
    print("📊 ===== EXPENSE INSIGHTS =====\n")
    
    # Top category
    top_category = df.groupby('category')['amount'].sum().idxmax()
    print(f"🔹 Top Category: {top_category}")
    
    # Average spend
    avg_spend = df['amount'].mean()
    print(f"🔹 Average Spend: ₹{avg_spend:.2f}")
    
    # Total spend
    total_spend = df['amount'].sum()
    print(f"🔹 Total Spend: ₹{total_spend}")
    
    # Anomaly summary
    anomalies = df[df['anomaly'] == -1]
    
    print(f"\n🚨 Total Anomalies: {len(anomalies)}")
    
    print("\n🔹 Anomalies by Category:")
    print(anomalies['category'].value_counts())
    
    print("\n🔹 Top 5 Highest Expenses:")
    print(df.sort_values(by='amount', ascending=False).head(5)[['date','amount','category']])
    
    
    
generate_insights(df)